In [9]:
import streamlit as st

import os

os.chdir("/home/vboxuser/Desktop/data_science/israel-public-sentiment")

os.environ["DATABRICKS_TOKEN"] = st.secrets["DATABRICKS_TOKEN"]

In [14]:
import os
import requests
import numpy as np
import pandas as pd
import json

def create_tf_serving_json(data):
    return {'inputs': {name: data[name].tolist() for name in data.keys()} if isinstance(data, dict) else data.tolist()}

def get_predictions(dataset):
    url = 'https://dbc-7e6efb50-7ae7.cloud.databricks.com/serving-endpoints/israel-public-sentiment-forecaster/invocations'
    headers = {'Authorization': f'Bearer {os.environ.get("DATABRICKS_TOKEN")}', 'Content-Type': 'application/json'}
    ds_dict = {'dataframe_split': dataset.to_dict(orient='split')} if isinstance(dataset, pd.DataFrame) else create_tf_serving_json(dataset)
    data_json = json.dumps(ds_dict, allow_nan=True)
    response = requests.request(method='POST', headers=headers, url=url, data=data_json)
    if response.status_code != 200:
        raise Exception(f'Request failed with status {response.status_code}, {response.text}')
    return response.json()

In [15]:
dataset = np.random.uniform(0, 1, (2, 2))
dataset = pd.DataFrame(dataset, columns=['GoldsteinScale', 'NumMentions'])

get_predictions(dataset)

{'predictions': [[-4.824629147511907,
   -5.406441848280612,
   -4.918072615456508,
   -4.822805521646602,
   -4.3088868420939],
  [-4.7282249229724025,
   -5.4819116615011225,
   -5.167725564445797,
   -4.903972516299897,
   -4.32320817593614]]}